![hslu_logo.png](img/hslu_logo.png)

## Week 3

<hr style="border:1px solid black">


# Excercise: Model Zoo of PyTorch
---
---
This excercise is to illustrate how to use a predefined model from the so-called `model zoo`of PyTorch

### Import necessary packages

In [ ]:
import torch
import torchvision
import numpy as np
import matplotlib.pyplot as plt

from utils import plot_img

#### List all available models

In [ ]:
all_models = torchvision.models.list_models()
for mod in range(len(all_models)):
    print(all_models[mod])

#### Select the model
We use vgg16 due to its yet simple architecture

In [ ]:
#download the model and the weights 
from torchvision.models import vgg16, VGG16_Weights

#### Prepare the model
We use the default weights and set up the model. Some models require to switch between `eval()` and `train()` 

Note that the architecture consists of two parts:
1. `(features)`<br>
   The subsequent convolutional and pooling layers
3. `(classifier)`<br>
   The two dense layers for the classification

In [ ]:
weights = VGG16_Weights.DEFAULT
vgg16_model = vgg16(weights=weights)
vgg16_model.eval()

#### Show the number of parameters
Note the predominant number of parameters of the first dense layer `classifier.0.weight 102760448`

In [ ]:
tot_param = []
cum_param = 0
for name, parameter in vgg16_model.named_parameters():
    print(name, parameter.numel())
    cum_param += parameter.numel()
    tot_param = tot_param + [cum_param]

plt.plot(tot_param, 'bo-')
plt.show()

#### Load an image

Note the shape with the channels at the first position

In [ ]:
img = torchvision.io.read_image('./sample_img/cat.jpg')
print(img.shape)
plot_img(torch.movedim(img, [0],[2]))

#### Transform the image

*Before using the pre-trained models, one must preprocess the image (resize with right resolution/interpolation, apply inference transforms, rescale the values etc). There is no standard way to do this as it depends on how a given model was trained. It can vary across model families, variants or even weight versions. Using the correct preprocessing method is critical and failing to do so may lead to decreased accuracy or incorrect outputs.*
[[ref](https://pytorch.org/vision/stable/models.html#using-the-pre-trained-models)]

The preprocessing is weight specific (a model can have different weight configurations) and therefore the weights (through the method `transforms()`) acutally contain the information on the precise transformation required. 

(The parameter `antialias=True` is only to avoid a PyTorch warning for future compatibility)

In [ ]:
img_trans = weights.transforms(antialias=True)(img)
print('\noriginal shape: ', img.shape)
print('transform shape: ', img_trans.shape)
print('\noriginal type: ', img.dtype)
print('transform type: ', img_trans.dtype)
print('\noriginal range: [', torch.min(img).item(), ',', torch.max(img).item(), ']')
print('transform range: [', torch.min(img_trans).item(), ',', torch.max(img_trans).item(), ']')

#### Classify the image

In [ ]:
#we have to add the batch dimension (only single image)
batch = img_trans.unsqueeze(0)
print(batch.shape)

#apply the model (including the missing softmax; this only influences the normalisation of the output, 
#not the order)
prediction = vgg16_model(batch).softmax(1)
class_id = torch.argmax(prediction, axis = 1)

#get score and class
score = prediction[0,class_id].item()
category_name = weights.meta["categories"][class_id]

print('score: ', score, '  category: ', category_name)